# 01 - Data Audit

**AurumMind AI - Phase 2: Exploratory Data Analysis**

This notebook performs a comprehensive audit of the ingested XAUUSD data.

## Objectives
1. Load validated data from Phase 1
2. Verify data quality and completeness
3. Identify any remaining issues
4. Document data characteristics
5. Generate audit report

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✓ Imports successful")

## 1. Load Sample Data

Load the sample XAUUSD data generated in Phase 1.

In [ ]:
# Load 1H data
data_1h_path = Path.cwd().parent / 'data' / 'raw' / 'samples' / 'XAUUSD_1h_sample.csv'
df_1h = pd.read_csv(data_1h_path)
df_1h['timestamp'] = pd.to_datetime(df_1h['timestamp'])

print(f"1H Data Loaded: {len(df_1h)} records")
print(f"Date Range: {df_1h['timestamp'].min()} to {df_1h['timestamp'].max()}")
print(f"\nColumns: {list(df_1h.columns)}")
df_1h.head()

In [ ]:
# Data overview
print("Data Info:")
print(df_1h.info())
print("\n" + "="*60)
print("\nBasic Statistics:")
df_1h.describe()

## 2. Data Quality Checks

In [ ]:
# Check for missing values
missing = df_1h.isnull().sum()
print("Missing Values:")
print(missing)
print(f"\nTotal Missing: {missing.sum()}")

# Check for duplicates
duplicates = df_1h.duplicated(subset=['timestamp']).sum()
print(f"\nDuplicate Timestamps: {duplicates}")

# Check data types
print("\nData Types:")
print(df_1h.dtypes)

In [ ]:
# OHLC relationship validation
invalid_high = (df_1h['high'] < df_1h[['open', 'close', 'low']].max(axis=1)).sum()
invalid_low = (df_1h['low'] > df_1h[['open', 'close', 'high']].min(axis=1)).sum()

print(f"Invalid High values: {invalid_high}")
print(f"Invalid Low values: {invalid_low}")
print(f"\nOHLC Validation: {'✓ PASSED' if (invalid_high == 0 and invalid_low == 0) else '✗ FAILED'}")

## 3. Time Series Continuity

In [ ]:
# Check timestamp continuity
df_1h_sorted = df_1h.sort_values('timestamp').reset_index(drop=True)
time_diffs = df_1h_sorted['timestamp'].diff()

expected_freq = pd.Timedelta(hours=1)
gaps = time_diffs[time_diffs != expected_freq].dropna()

print(f"Expected Frequency: {expected_freq}")
print(f"Number of Gaps: {len(gaps)}")

if len(gaps) > 0:
    print("\nGap Details:")
    for idx in gaps.index[:5]:  # Show first 5 gaps
        print(f"  Gap at index {idx}: {time_diffs.loc[idx]}")

## 4. Price Statistics

In [ ]:
# Price statistics
price_stats = {
    'Metric': [],
    'Open': [],
    'High': [],
    'Low': [],
    'Close': []
}

metrics = ['Mean', 'Median', 'Std', 'Min', 'Max', '25%', '75%']
for metric in metrics:
    price_stats['Metric'].append(metric)
    if metric == 'Mean':
        price_stats['Open'].append(df_1h['open'].mean())
        price_stats['High'].append(df_1h['high'].mean())
        price_stats['Low'].append(df_1h['low'].mean())
        price_stats['Close'].append(df_1h['close'].mean())
    elif metric == 'Median':
        price_stats['Open'].append(df_1h['open'].median())
        price_stats['High'].append(df_1h['high'].median())
        price_stats['Low'].append(df_1h['low'].median())
        price_stats['Close'].append(df_1h['close'].median())
    elif metric == 'Std':
        price_stats['Open'].append(df_1h['open'].std())
        price_stats['High'].append(df_1h['high'].std())
        price_stats['Low'].append(df_1h['low'].std())
        price_stats['Close'].append(df_1h['close'].std())
    elif metric == 'Min':
        price_stats['Open'].append(df_1h['open'].min())
        price_stats['High'].append(df_1h['high'].min())
        price_stats['Low'].append(df_1h['low'].min())
        price_stats['Close'].append(df_1h['close'].min())
    elif metric == 'Max':
        price_stats['Open'].append(df_1h['open'].max())
        price_stats['High'].append(df_1h['high'].max())
        price_stats['Low'].append(df_1h['low'].max())
        price_stats['Close'].append(df_1h['close'].max())
    elif metric == '25%':
        price_stats['Open'].append(df_1h['open'].quantile(0.25))
        price_stats['High'].append(df_1h['high'].quantile(0.25))
        price_stats['Low'].append(df_1h['low'].quantile(0.25))
        price_stats['Close'].append(df_1h['close'].quantile(0.25))
    elif metric == '75%':
        price_stats['Open'].append(df_1h['open'].quantile(0.75))
        price_stats['High'].append(df_1h['high'].quantile(0.75))
        price_stats['Low'].append(df_1h['low'].quantile(0.75))
        price_stats['Close'].append(df_1h['close'].quantile(0.75))

stats_df = pd.DataFrame(price_stats)
print("Price Statistics:")
stats_df

## 5. Volume Analysis

In [ ]:
print("Volume Statistics:")
print(f"Mean: {df_1h['volume'].mean():.2f}")
print(f"Median: {df_1h['volume'].median():.2f}")
print(f"Std: {df_1h['volume'].std():.2f}")
print(f"Min: {df_1h['volume'].min():.2f}")
print(f"Max: {df_1h['volume'].max():.2f}")

zero_volume = (df_1h['volume'] == 0).sum()
print(f"\nZero Volume Candles: {zero_volume} ({zero_volume/len(df_1h)*100:.2f}%)")

## 6. Data Audit Summary

In [ ]:
print("="*60)
print("DATA AUDIT SUMMARY")
print("="*60)
print(f"\nDataset: XAUUSD 1H")
print(f"Total Records: {len(df_1h)}")
print(f"Date Range: {df_1h['timestamp'].min()} to {df_1h['timestamp'].max()}")
print(f"Duration: {(df_1h['timestamp'].max() - df_1h['timestamp'].min()).days} days")
print(f"\nQuality Checks:")
print(f"  ✓ Missing Values: {missing.sum()}")
print(f"  ✓ Duplicate Timestamps: {duplicates}")
print(f"  ✓ Invalid OHLC: {invalid_high + invalid_low}")
print(f"  ✓ Time Gaps: {len(gaps)}")
print(f"\nPrice Range:")
print(f"  Lowest Low: ${df_1h['low'].min():.2f}")
print(f"  Highest High: ${df_1h['high'].max():.2f}")
print(f"  Range: ${df_1h['high'].max() - df_1h['low'].min():.2f}")
print(f"\n✓ Data audit complete")